# Reading and writing data with Pandas

> **Explain it like I am five:** Data can arrive in different lunch boxes—CSV, JSON, HTML, Excel, or Pickle. Pandas opens each box with a different reader, but the food inside becomes a DataFrame.

This notebook preserves the original JSON, web CSV, HTML-table, Excel, and Pickle examples while making the default path offline, reproducible, and safe for this repository.

## Learning goals

- read and write JSON, CSV, HTML tables, Excel, and Pickle;
- choose useful JSON orientations;
- handle paths, encodings, date parsing, chunks, and file round trips;
- understand which formats are portable, typed, safe, and compact.


In [1]:
import json
from io import StringIO
from pathlib import Path

import pandas as pd

print("Pandas version:", pd.__version__)


Pandas version: 3.0.3


In [2]:
from pathlib import Path

def data_path(filename):
    """Find a course data file whether Jupyter starts in the repo or lesson folder."""
    current = Path.cwd().resolve()
    lesson_parts = ("Complete-Python-Bootcamp-main", "10-Data Analysis With Python")
    candidates = [current / filename, current.joinpath(*lesson_parts, filename)]
    for parent in current.parents:
        candidates.extend([parent / filename, parent.joinpath(*lesson_parts, filename)])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {filename!r}. Start Jupyter inside this repository.")


## 0. Optional parsers

If an import error says a parser is missing, install it once in the active notebook environment:

```python
%pip install lxml html5lib beautifulsoup4 openpyxl
```

The original notebook used four `!pip install ...` commands. Installation is setup, so it should not run every time the lesson runs.


## 1. JSON: nested labeled data

JSON stores dictionaries, lists, strings, numbers, booleans, and null values. `StringIO` lets Pandas read text as if it were a small file.


In [3]:
Data = '{"employee_name": "James", "email": "james@gmail.com", "job_profile": [{"title1":"Team Lead", "title2":"Sr. Developer"}]}'
df=pd.read_json(StringIO(Data))
df


,employee_name,email,job_profile
0,James,james@gmail.com,"{'title1': 'Team Lead', 'title2': 'Sr. Develop..."


The nested `job_profile` remains an object inside one cell. Use `json_normalize` when you want nested keys flattened into columns.


In [4]:
parsed = json.loads(Data)
normalized_employee = pd.json_normalize(parsed, sep='_')
normalized_employee


,employee_name,email,job_profile
0,James,james@gmail.com,"[{'title1': 'Team Lead', 'title2': 'Sr. Develo..."


## 2. JSON orientations

Orientation describes how rows and columns are arranged in JSON.

- default `columns`: column-first mapping;
- `index`: row-label-first mapping;
- `records`: list of row dictionaries, common for APIs.


In [5]:
print("default/columns:\n", df.to_json())
print("\nindex:\n", df.to_json(orient='index'))
print("\nrecords:\n", df.to_json(orient='records'))


default/columns:
 {"employee_name":{"0":"James"},"email":{"0":"james@gmail.com"},"job_profile":{"0":{"title1":"Team Lead","title2":"Sr. Developer"}}}

index:
 {"0":{"employee_name":"James","email":"james@gmail.com","job_profile":{"title1":"Team Lead","title2":"Sr. Developer"}}}

records:
 [{"employee_name":"James","email":"james@gmail.com","job_profile":{"title1":"Team Lead","title2":"Sr. Developer"}}]


In [6]:
records_json = df.to_json(orient='records')
round_trip_json = pd.read_json(StringIO(records_json), orient='records')
print(round_trip_json)


  employee_name            email  \
0         James  james@gmail.com   

                                         job_profile  
0  {'title1': 'Team Lead', 'title2': 'Sr. Develop...  


## 3. CSV: simple, portable text tables

CSV is easy to share but does not preserve rich types. Read options tell Pandas how to interpret dates, missing markers, separators, and encodings.


In [7]:
sales = pd.read_csv(
    data_path('sales_data.csv'),
    parse_dates=['Date'],
    dtype={'Transaction ID': 'int64'}
)
print(sales.head())
print(sales.dtypes)


   Transaction ID       Date Product Category             Product Name  \
0           10001 2024-01-01      Electronics            iPhone 14 Pro   
1           10002 2024-01-02  Home Appliances         Dyson V11 Vacuum   
2           10003 2024-01-03         Clothing         Levi's 501 Jeans   
3           10004 2024-01-04            Books        The Da Vinci Code   
4           10005 2024-01-05  Beauty Products  Neutrogena Skincare Set   

   Units Sold  Unit Price  Total Revenue         Region Payment Method  
0           2      999.99        1999.98  North America    Credit Card  
1           1      499.99         499.99         Europe         PayPal  
2           3       69.99         209.97           Asia     Debit Card  
3           4       15.99          63.96  North America    Credit Card  
4           1       89.99          89.99         Europe         PayPal  
Transaction ID               int64
Date                datetime64[us]
Product Category               str
Product Name

### Original online Wine example, with an offline default

Live websites can change or be unavailable. Set `RUN_LIVE_WEB_EXAMPLES = True` when internet access is available. Otherwise, this cell reads the repository's saved copy.


In [8]:
RUN_LIVE_WEB_EXAMPLES = False
wine_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine/wine.data"
wine_columns = [
    'Class', 'Alcohol', 'Malic_Acid', 'Ash', 'Alcalinity_of_Ash',
    'Magnesium', 'Total_Phenols', 'Flavanoids', 'Nonflavanoid_Phenols',
    'Proanthocyanins', 'Color_Intensity', 'Hue',
    'OD280_OD315_Diluted_Wines', 'Proline'
]

if RUN_LIVE_WEB_EXAMPLES:
    wine_df = pd.read_csv(wine_url, header=None, names=wine_columns)
else:
    wine_df = pd.read_csv(data_path('wine.csv'), index_col=0)
    wine_df.columns = wine_columns

wine_df.head()


,Class,Alcohol,Malic_Acid,Ash,Alcalinity_of_Ash,Magnesium,Total_Phenols,Flavanoids,Nonflavanoid_Phenols,Proanthocyanins,Color_Intensity,Hue,OD280_OD315_Diluted_Wines,Proline
0,1,14.23,1.71,2.43,15.6,127,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065
1,1,13.20,1.78,2.14,11.2,100,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050
2,1,13.16,2.36,2.67,18.6,101,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185
3,1,14.37,1.95,2.50,16.8,113,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480
4,1,13.24,2.59,2.87,21.0,118,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735


The original `df.to_csv("wine.csv")` writes the index by default, which created the saved `Unnamed: 0` column. Usually use `index=False` unless row labels carry meaning.


In [9]:
export_path = data_path('wine.csv').with_name('wine_roundtrip_demo.csv')
wine_df.to_csv(export_path, index=False)

try:
    wine_round_trip = pd.read_csv(export_path)
    print(wine_round_trip.head(3))
    print("same shape:", wine_round_trip.shape == wine_df.shape)
finally:
    export_path.unlink(missing_ok=True)


   Class  Alcohol  Malic_Acid   Ash  Alcalinity_of_Ash  Magnesium  \
0      1    14.23        1.71  2.43               15.6        127   
1      1    13.20        1.78  2.14               11.2        100   
2      1    13.16        2.36  2.67               18.6        101   

   Total_Phenols  Flavanoids  Nonflavanoid_Phenols  Proanthocyanins  \
0           2.80        3.06                  0.28             2.29   
1           2.65        2.76                  0.26             1.28   
2           2.80        3.24                  0.30             2.81   

   Color_Intensity   Hue  OD280_OD315_Diluted_Wines  Proline  
0             5.64  1.04                       3.92     1065  
1             4.38  1.05                       3.40     1050  
2             5.68  1.03                       3.17     1185  
same shape: True


## 4. Read large CSV files in chunks

A chunk is one manageable slice of a large file. This avoids loading every row into memory at once.


In [10]:
revenue_total = 0.0
row_count = 0

for chunk in pd.read_csv(data_path('sales_data.csv'), chunksize=50):
    revenue_total += chunk['Total Revenue'].sum()
    row_count += len(chunk)

print("rows:", row_count)
print("total revenue:", round(revenue_total, 2))


rows: 240
total revenue: 80567.85


## 5. HTML tables

`pd.read_html` returns a **list** because one page can contain several tables. It reads tables, not arbitrary page text.

The original examples use FDIC and Wikipedia pages. The default below uses a tiny local HTML table, so the notebook works offline.


In [11]:
fdic_url="https://www.fdic.gov/resources/resolutions/bank-failures/failed-bank-list/"
country_code_url="https://en.wikipedia.org/wiki/Mobile_country_code"

local_html = """
<table>
  <thead><tr><th>Country</th><th>Mobile code</th></tr></thead>
  <tbody>
    <tr><td>India</td><td>404</td></tr>
    <tr><td>United States</td><td>310</td></tr>
    <tr><td>United Kingdom</td><td>234</td></tr>
  </tbody>
</table>
"""

if RUN_LIVE_WEB_EXAMPLES:
    fdic_tables = pd.read_html(fdic_url)
    country_codes = pd.read_html(country_code_url, match="Country", header=0)[0]
    print("FDIC tables found:", len(fdic_tables))
else:
    # Offline equivalent of the tiny table above. This keeps the core notebook
    # runnable even when optional HTML parser packages are not installed.
    country_codes = pd.DataFrame({
        'Country': ['India', 'United States', 'United Kingdom'],
        'Mobile code': [404, 310, 234]
    })

country_codes


,Country,Mobile code
0,India,404
1,United States,310
2,United Kingdom,234


Websites may change markup or block automated requests. For dependable projects, save a permitted source snapshot, record its retrieval date, and validate expected columns.


## 6. Excel workbooks

Excel can store multiple sheets, formatting, and typed cells. `sheet_name=None` reads every sheet into a dictionary of DataFrames.


In [12]:
df_excel=pd.read_excel(data_path('data.xlsx'))
print(df_excel)


    Name  Age
0  Krish   32
1   Jack   34
2   John   31


In [13]:
sample_sheets = pd.read_excel(data_path('sample_data.xlsx'), sheet_name=None)
print("sheets:", list(sample_sheets))
for sheet_name, table in sample_sheets.items():
    print(f"\n{sheet_name} shape: {table.shape}")
    print(table.head())


sheets: ['People', 'Sales Sample']

People shape: (4, 4)
    Name  Age       City     Joined
0  Krish   32  Bangalore 2024-01-15
1   Jack   34   New York 2024-02-10
2   John   31    Florida 2024-03-05
3   Maya   29       Pune 2024-04-12

Sales Sample shape: (6, 5)
        Date   Product  Sales Region  Units
0 2024-01-01  Product1    754   East      4
1 2024-01-02  Product3    110  North      1
2 2024-01-03  Product2    398   East      2
3 2024-01-04  Product1    625   West      3
4 2024-01-05  Product3    482  South      2


Select columns and types while reading when that reduces ambiguity:


In [14]:
selected_people = pd.read_excel(
    data_path('data.xlsx'),
    usecols=['Name', 'Age'],
    dtype={'Name': 'string', 'Age': 'int64'}
)
print(selected_people.dtypes)
print(selected_people)


Name    string
Age      int64
dtype: object
    Name  Age
0  Krish   32
1   Jack   34
2   John   31


## 7. Pickle: Python-specific snapshots

Pickle preserves Python/Pandas objects accurately, but it is Python-specific and not safe for untrusted input. Loading a malicious pickle can execute code.

The original workbook is converted to `df_excel` and read back below.


In [15]:
pickle_path = data_path('df_excel')
df_excel.to_pickle(pickle_path)
restored_excel = pd.read_pickle(pickle_path)
restored_excel


,Name,Age
0,Krish,32
1,Jack,34
2,John,31


**Safety rule:** only call `read_pickle` on files you created or fully trust. Use CSV, JSON, Excel, or Parquet for safer data exchange.


## 8. Format comparison

| Format | Best for | Keeps types? | Human-readable? | Main caution |
|---|---|---:|---:|---|
| CSV | simple exchange | limited | yes | dates/types must be restored |
| JSON | APIs and nested data | partly | yes | orientation and nesting matter |
| HTML | extracting page tables | limited | yes | pages change |
| Excel | business workbooks | good | yes | sheet names and engines matter |
| Pickle | trusted Python snapshots | excellent | no | never load untrusted files |
| Parquet | analytical storage | excellent | no | requires a Parquet engine |


## 9. Validate after reading

Reading successfully does not prove the data is correct. Check required columns, row counts, ranges, uniqueness, and missing values.


In [16]:
required_columns = {'Date', 'Product Category', 'Total Revenue', 'Region'}
missing_columns = required_columns.difference(sales.columns)

assert not missing_columns, f"Missing columns: {sorted(missing_columns)}"
assert sales['Transaction ID'].is_unique, "Transaction IDs should be unique"
assert sales['Total Revenue'].ge(0).all(), "Revenue should not be negative"

print("Validation checks passed for", len(sales), "rows.")


Validation checks passed for 240 rows.


## 10. Common mistakes

- relying on the current working directory instead of a resolved path;
- saving a CSV index accidentally and getting `Unnamed: 0` later;
- trusting inferred dates, IDs, or missing markers without checking;
- assuming `read_html` returns one DataFrame instead of a list;
- depending on live websites for a core lesson;
- loading an untrusted Pickle file;
- reading a huge CSV at once when chunks would work;
- writing a file without immediately reading it back to verify the round trip.


## 11. Mini practice

1. Read `sales1_data.csv`, parsing its date column.
2. Write it to JSON using records orientation, then read it back from `StringIO`.
3. Confirm that required columns exist and sales are nonnegative when present.


In [17]:
practice = pd.read_csv(data_path('sales1_data.csv'), parse_dates=['Date'])
practice_json = practice.to_json(orient='records', date_format='iso')
practice_round_trip = pd.read_json(StringIO(practice_json), orient='records')

assert {'Date', 'Product', 'Sales', 'Region'}.issubset(practice_round_trip.columns)
assert practice_round_trip['Sales'].dropna().ge(0).all()
print(practice_round_trip.head())


        Date   Product  Sales Region
0 2023-01-01  Product3  738.0   West
1 2023-01-02  Product2  868.0  North
2 2023-01-03  Product2  554.0   West
3 2023-01-04  Product1  618.0  South
4 2023-01-05  Product3  501.0   East


## Easy revision cheat sheet

| Need | Pattern |
|---|---|
| CSV | `pd.read_csv(path, parse_dates=[...])` |
| JSON text | `pd.read_json(StringIO(text))` |
| Flatten JSON | `pd.json_normalize(object)` |
| JSON rows | `df.to_json(orient='records')` |
| HTML tables | `pd.read_html(url_or_text)` → list |
| Excel sheet | `pd.read_excel(path, sheet_name='Sheet')` |
| Every Excel sheet | `pd.read_excel(path, sheet_name=None)` |
| Trusted snapshot | `df.to_pickle(path)`, `pd.read_pickle(path)` |
| Large CSV | `pd.read_csv(path, chunksize=n)` |
| Avoid CSV index | `df.to_csv(path, index=False)` |
| Portable path | use `pathlib.Path` and verify `.exists()` |
| Validate schema | compare `required` with `df.columns` |

**Memory trick:** A reader opens the lunch box; read options explain its labels; validation checks that the expected lunch actually arrived.
